# ReviveTech 2026 — Machine Learning Analysis & Blind Test Evaluations

Notebook gerado a partir do pipeline de ML do projeto: amostragem estratificada, visualização da massa de dados, split cego 80/20, regressão logística, árvore CART, e avaliação em teste cego (matrizes de confusão, curva de aprendizado, ROC, importância de atributos e árvore de decisão).

## Célula 1 — Importações, Estilo Visual e Carga dos Dados

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Configuração do estilo visual do Matplotlib e Seaborn
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

# Carregamento do dataset e tratamento de limites não-negativos
df = pd.read_csv('planilha_fogos_consolidados.csv')

for col in ['frp', 'precipitacao', 'numero_dias_sem_chuva', 'qtd_focos', 'risco_fogo']:
    if col in df.columns:
        df[col] = df[col].clip(lower=0.0)

print(f"Total de registros brutos carregados: {len(df):,}")
df.head()

Error: No connection selected.

## Célula 2 — Amostragem Estratificada (bioma × faixa de risco)

Reduz o volume de trabalho sem distorcer as proporções originais do dataset.
Roda antes de qualquer gráfico ou split, então todo o resto do pipeline
(visualizações, treino, validação cega) já opera sobre a amostra reduzida.

In [2]:
SAMPLE_SIZE = 3000  # ajuste conforme o tamanho do dataset e o tempo disponível

if len(df) > SAMPLE_SIZE:
    df['risco_bin'] = pd.cut(
        df['risco_fogo'], bins=[-0.01, 0.3, 0.7, 1.01],
        labels=['baixo', 'moderado', 'critico']
    )
    estrato = df['bioma'].astype(str) + '-' + df['risco_bin'].astype(str)

    df, _ = train_test_split(
        df, train_size=SAMPLE_SIZE,
        stratify=estrato, random_state=42
    )
    df = df.drop(columns='risco_bin').reset_index(drop=True)

print(f"Amostra de trabalho: {len(df):,} registros "
      f"(estratificada por bioma e faixa de risco, proporcional ao dataset original)")

Error: No connection selected.

## Célula 3 — Visualização da Massa de Dados de Treino (Matplotlib & Seaborn)

In [3]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Training Data Mass — Fire Hotspots by Biome & Environmental Factors', fontsize=16, fontweight='bold', y=0.98)

# Ordenação dos Biomas
biome_order = df['bioma'].value_counts().index

# Plot 1: Distribuição de Focos por Bioma
sns.countplot(data=df, x='bioma', palette='Set2', ax=axes[0, 0], order=biome_order)
axes[0, 0].set_title('Hotspot Distribution by Biome', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Biome', fontsize=11)
axes[0, 0].set_ylabel('Record Count', fontsize=11)
axes[0, 0].set_ylim(bottom=0)
axes[0, 0].tick_params(axis='x', rotation=15)

for p in axes[0, 0].patches:
    height = p.get_height()
    if height > 0:
        axes[0, 0].annotate(f'{height:,}', (p.get_x() + p.get_width() / 2.0, height / 2.0),
                            ha='center', va='center', fontsize=10, color='black', fontweight='bold')

# Plot 2: Distribuição de FRP (Fire Radiative Power) por Bioma
sns.boxplot(data=df, x='bioma', y='frp', palette='Set2', ax=axes[0, 1], order=biome_order)
axes[0, 1].set_title('Fire Intensity (FRP) by Biome', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Biome', fontsize=11)
axes[0, 1].set_ylabel('Fire Radiative Power (FRP, MW)', fontsize=11)
axes[0, 1].set_ylim(bottom=0)
axes[0, 1].tick_params(axis='x', rotation=15)

# Plot 3: Dias Sem Chuva vs. FRP (Fire Radiative Power)
sns.scatterplot(data=df, x='numero_dias_sem_chuva', y='frp', hue='bioma', palette='Set2', alpha=0.7, ax=axes[1, 0])
axes[1, 0].set_title('Days Without Rain vs. Fire Radiative Power (FRP)', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Number of Days Without Rain', fontsize=11)
axes[1, 0].set_ylabel('Fire Radiative Power (FRP, MW)', fontsize=11)
axes[1, 0].set_xlim(left=0)
axes[1, 0].set_ylim(bottom=0)
axes[1, 0].legend(title='Biome', loc='upper left')

# Plot 4: Distribuição da Variável Alvo (Risco de Fogo)
risk_counts = df['risco_fogo'].value_counts().sort_index()
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#9b59b6']
axes[1, 1].bar(risk_counts.index.astype(str), risk_counts.values, color=colors[:len(risk_counts)], edgecolor='black', alpha=0.85)
axes[1, 1].set_title('Target Variable Distribution: Fire Risk Class', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Risk Level / Class', fontsize=11)
axes[1, 1].set_ylabel('Record Count', fontsize=11)
axes[1, 1].set_ylim(bottom=0)

for p in axes[1, 1].patches:
    height = p.get_height()
    if height > 0:
        axes[1, 1].annotate(f'{height:,}', (p.get_x() + p.get_width() / 2.0, height / 2.0),
                            ha='center', va='center', fontsize=10, color='white', fontweight='bold')

plt.tight_layout()
plt.show()

Error: No connection selected.

## Célula 4 — Divisão Cega dos Dados (Blind Test Split 80/20)

In [5]:
frp_median = df['frp'].median()
df['severity_class'] = ((df['frp'] > frp_median) | (df['risco_fogo'] >= 0.8)).astype(int)

feature_cols = ['lat_fogo', 'lon_fogo', 'qtd_focos', 'frp', 'precipitacao', 'numero_dias_sem_chuva', 'risco_fogo']
X = df[feature_cols].fillna(0)
y = df['severity_class']

# Divisão estratificada (80% treino, 20% teste cego)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Dataset (80%): {X_train.shape[0]:,} samples")
print(f"Blind Test Dataset (20%): {X_test.shape[0]:,} samples")
print(f"Positive class (Critical) ratio in Train: {y_train.mean():.2%}")
print(f"Positive class (Critical) ratio in Blind Test: {y_test.mean():.2%}")

Error: No connection selected.

## Célula 5 — Treinamento dos Modelos (Regressão Logística & Árvore CART)

In [6]:
# 1. Regressão Logística
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

# 2. Árvore de Decisão CART
cart_model = DecisionTreeClassifier(max_depth=3, min_samples_split=10, random_state=42)
cart_model.fit(X_train, y_train)
y_pred_cart = cart_model.predict(X_test)
y_prob_cart = cart_model.predict_proba(X_test)[:, 1]

# Avaliação nos testes cegos
acc_lr = accuracy_score(y_test, y_pred_lr)
acc_cart = accuracy_score(y_test, y_pred_cart)
f1_lr = f1_score(y_test, y_pred_lr)
f1_cart = f1_score(y_test, y_pred_cart)

print("=== BLIND TEST RESULTS ===")
print(f"Logistic Regression -> Accuracy: {acc_lr:.4f} | F1-Score: {f1_lr:.4f}")
print(f"Clean CART Tree     -> Accuracy: {acc_cart:.4f} | F1-Score: {f1_cart:.4f}")

Error: No connection selected.

## Célula 6 — Matrizes de Confusão Comparativas (Heatmaps)

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz Regressão Logística
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Moderate (0)', 'Critical (1)'], yticklabels=['Moderate (0)', 'Critical (1)'])
axes[0].set_title(f'Logistic Regression (Accuracy: {acc_lr:.2%})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('Actual Label (Blind Test)')

# Matriz Árvore CART
cm_cart = confusion_matrix(y_test, y_pred_cart)
sns.heatmap(cm_cart, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Moderate (0)', 'Critical (1)'], yticklabels=['Moderate (0)', 'Critical (1)'])
axes[1].set_title(f'Clean CART Tree (Accuracy: {acc_cart:.2%})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('Actual Label (Blind Test)')

plt.tight_layout()
plt.show()

Error: No connection selected.

## Célula 7 — Curva de Aprendizado e Curva ROC (Validação Cega)

In [8]:
train_sizes, train_scores_lr, test_scores_lr = learning_curve(
    lr_model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)
train_sizes, train_scores_cart, test_scores_cart = learning_curve(
    cart_model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)

mean_test_lr = np.mean(test_scores_lr, axis=1)
mean_test_cart = np.mean(test_scores_cart, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Evolução da Acurácia nos Treinos Cegos
axes[0].plot(train_sizes, mean_test_lr, 'o-', color='#2980b9', linewidth=2.5, label='Logistic Regression (Blind Validation)')
axes[0].plot(train_sizes, mean_test_cart, 's-', color='#27ae60', linewidth=2.5, label='CART Decision Tree (Blind Validation)')
axes[0].set_title('Accuracy Growth in Blind Tests by Sample Size', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Training Sample Size', fontsize=11)
axes[0].set_ylabel('Mean Validation Accuracy', fontsize=11)
axes[0].set_xlim(left=0)
axes[0].set_ylim(0.5, 1.01)
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, linestyle='--', alpha=0.6)

# Plot 2: Curva ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_cart, tpr_cart, _ = roc_curve(y_test, y_prob_cart)
auc_lr = auc(fpr_lr, tpr_lr)
auc_cart = auc(fpr_cart, tpr_cart)

axes[1].plot(fpr_lr, tpr_lr, color='#2980b9', linewidth=2, label=f'Logistic Regression (AUC = {auc_lr:.3f})')
axes[1].plot(fpr_cart, tpr_cart, color='#27ae60', linewidth=2, label=f'CART Decision Tree (AUC = {auc_cart:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Classifier (AUC = 0.500)')
axes[1].set_title('ROC Curve — Blind Test Performance Comparison', fontsize=12, fontweight='bold')
axes[1].set_xlabel('False Positive Rate (FPR)', fontsize=11)
axes[1].set_ylabel('True Positive Rate (TPR)', fontsize=11)
axes[1].set_xlim(0.0, 1.0)
axes[1].set_ylim(0.0, 1.02)
axes[1].legend(loc='lower right', fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

Error: No connection selected.

## Célula 8 — Predições de ML (Risco Moderado vs Crítico) e Importância dos Atributos

In [9]:
df['prob_critical_risk'] = cart_model.predict_proba(X)[:, 1]
df['predicted_category'] = np.where(df['prob_critical_risk'] >= 0.7, 'Critical/High',
                             np.where(df['prob_critical_risk'] >= 0.3, 'Moderate', 'Low'))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Dispersão de Risco Predito
palette_risk = {'Low': '#2ecc71', 'Moderate': '#f39c12', 'Critical/High': '#e74c3c'}
sns.scatterplot(
    data=df, x='numero_dias_sem_chuva', y='frp',
    hue='predicted_category', palette=palette_risk, alpha=0.8, s=50, ax=axes[0]
)
axes[0].set_title('Machine Learning Predictions: Moderate vs. Critical Risk', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Days Without Rain', fontsize=11)
axes[0].set_ylabel('Fire Radiative Power (FRP, MW)', fontsize=11)
axes[0].set_xlim(left=0)
axes[0].set_ylim(bottom=0)
axes[0].legend(title='Predicted Risk', loc='upper left')

# Plot 2: Importância dos Atributos no Modelo CART
feature_labels_en = {
    'frp': 'Fire Radiative Power (FRP, MW)',
    'risco_fogo': 'Fire Risk Index',
    'numero_dias_sem_chuva': 'Days Without Rain',
    'precipitacao': 'Precipitation',
    'qtd_focos': 'Hotspot Count',
    'lat_fogo': 'Latitude',
    'lon_fogo': 'Longitude'
}

importances = cart_model.feature_importances_
indices = np.argsort(importances)[::-1]
sorted_features_en = [feature_labels_en.get(feature_cols[i], feature_cols[i]) for i in indices]
sorted_importances = importances[indices]

axes[1].barh(sorted_features_en[::-1], sorted_importances[::-1], color='#34495e', edgecolor='black', alpha=0.85)
axes[1].set_title('Feature Importances in CART Decision Tree', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Relative Importance Weight', fontsize=11)
axes[1].set_ylabel('Input Features', fontsize=11)
axes[1].set_xlim(left=0, right=max(sorted_importances) * 1.15)

for i, v in enumerate(sorted_importances[::-1]):
    axes[1].text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

Error: No connection selected.

## Célula 9 — Diagrama Limpo da Árvore CART e Regras em Texto

In [10]:
fig, ax = plt.subplots(figsize=(20, 8), dpi=200)

plot_tree(
    cart_model,
    max_depth=3,
    feature_names=[feature_labels_en[c] for c in feature_cols],
    class_names=['Moderate Risk', 'Critical Risk'],
    filled=True,
    rounded=True,
    fontsize=10,
    precision=1,
    impurity=False,
    ax=ax
)

plt.title('CART Decision Tree — AI Decision Rules & Split Thresholds', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Regras Decisórias em Texto
print("=== EXACT CART AI DECISION RULES ===")
rules_text = export_text(cart_model, feature_names=[feature_labels_en[c] for c in feature_cols])
print(rules_text)

Error: No connection selected.